# 04：前六個月的 thread–PERMNO network

每個目標月份 M 使用 **M−6 至 M−1** 的留言。例如十月 network 使用四月至九月，不含十月留言。先按留言月份篩選，再將同一 thread 在整個六個月視窗內提及的 PERMNO 合併；不同留言、不同月份的 PERMNO 也可以因同一 thread 建立連結。

- 無方向網路；同一 thread 內不同 PERMNO 兩兩連結，不建立自我連結。同一 PERMNO 的重複提及不增加權重。
- `n_common_threads` 是視窗內共同提及該 PERMNO pair 的不同 thread 數；只要 n_common_threads ≥ 1 就有連結。
- 節點包含視窗內有留言提及的全部 PERMNO，保留孤立節點。`n_threads` 是該 PERMNO 所屬不同有效 thread 數，`degree` 是鄰居數，`strength_raw` 是共同 thread 數總和；`strength` 是相鄰 `log1p(n_common_threads)` 總和。
- 缺少月份者不納入視窗；有月份但缺少 thread ID 者可貢獻節點，不能建立連結。缺少或無效 PERMNO 時直接停止，不猜測配對。
- 預設使用最新成功的 02_02 run，可用 `INPUT` 指定來源。核對成功摘要與實際筆數、PERMNO 排除名單。
- 輸出月份從資料最早月份 + 6 開始，至最晚月份 + 1，連續逐月建立；不建立不足六個月歷史的早期網路。六個月中無留言的月份視為零活動。這表示觀察範圍涵蓋六個日曆月，無法保證原始資料在每個月都完整。
- 每次執行產生新 UTC 目錄，逐月輸出 `edges.parquet`、`nodes.parquet`，另存 `monthly_network_summary.csv` 與 `run_summary.json`，不修改前面的資料。

直接使用留言時間點已配對的 PERMNO。同一 thread 的 ticker 改名或別名先按 PERMNO 合併，再建立不同證券的連結；不在建網後猜測 ticker–PERMNO，不產生同證券 self-loop。02_02 必須已切換至 PERMNO 審核。

Edge 欄位：`permno_i, permno_j, n_common_threads, weight`，其中 `permno_i < permno_j`，`weight = log1p(n_common_threads)`。每月股票配對唯一，輸出前明確檢查 self-loop、方向與重複配對。


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
from itertools import combinations
import hashlib, json, time, math
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'Data/process').is_dir())
INPUT = None  # 可指定 02_02 的 comment_ticker_cleaned.parquet 路徑
WINDOW_MONTHS = 6
if INPUT is None:
    candidates = []
    for manifest in (ROOT / 'Data/process/02_02_filtered_comment_ticker').glob('*/run_summary.json'):
        info = json.loads(manifest.read_text(encoding='utf-8'))
        source = manifest.parent / 'comment_ticker_cleaned.parquet'
        if info.get('status') == 'complete' and source.is_file():
            candidates.append((manifest.parent.name, source, manifest, info))
    if not candidates:
        raise FileNotFoundError('找不到成功的 02_02 輸出')
    _, SOURCE, MANIFEST, upstream = max(candidates, key=lambda x: x[0])
else:
    SOURCE = Path(INPUT)
    if not SOURCE.is_absolute(): SOURCE = ROOT / SOURCE
    MANIFEST = SOURCE.parent / 'run_summary.json'
    upstream = json.loads(MANIFEST.read_text(encoding='utf-8'))
    if upstream.get('status') != 'complete': raise ValueError('02_02 尚未成功完成')
if upstream.get('review_unit') != 'permno' or 'permno' not in pq.read_schema(SOURCE).names:
    raise ValueError('02_02 仍是舊 ticker 輸出；請先執行新版 02_02，禁止猜測 PERMNO。')
source_stat = (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns)
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()
source_hash = sha256_file(SOURCE)
OUT = ROOT / 'Data/process/04_monthly_ticker_network' / datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_UTC')
OUT.mkdir(parents=True, exist_ok=False)
started = time.perf_counter()
print('來源：', SOURCE)
print('輸出：', OUT)


來源： c:\Users\eagle\RedditStock\Data\process\02_02_filtered_comment_ticker\2026-10-13_05-51-49_UTC\comment_ticker_cleaned.parquet
輸出： c:\Users\eagle\RedditStock\Data\process\04_monthly_ticker_network\2026-10-13_05-59-46_UTC


In [2]:
# 分批建立 month → thread → distinct PERMNO，月份來自留言本身。
monthly_threads = defaultdict(lambda: defaultdict(set))
monthly_nodes = defaultdict(set)
input_rows = missing_month_rows = missing_thread_rows = missing_permno_rows = 0
excluded = set(upstream['excluded_permnos'])
parquet = pq.ParquetFile(SOURCE)
for batch in parquet.iter_batches(batch_size=100_000, columns=['year_month', 'thread_id', 'permno']):
    frame = batch.to_pandas()
    input_rows += len(frame)
    missing_month = frame.year_month.isna() | frame.year_month.eq('')
    missing_thread = frame.thread_id.isna() | frame.thread_id.str.strip().eq('')
    missing_permno = frame.permno.isna()
    if missing_permno.any() or not frame.permno.gt(0).all(): raise ValueError('無效 PERMNO')
    if not frame.permno.mod(1).eq(0).all(): raise ValueError('PERMNO 必須為整數')
    frame['permno'] = frame.permno.astype('int64')
    missing_month_rows += int(missing_month.sum())
    missing_thread_rows += int(missing_thread.sum())
    missing_permno_rows += int(missing_permno.sum())
    valid = frame.loc[~missing_month & ~missing_permno]
    if not valid.year_month.str.fullmatch(r'\d{4}-(0[1-9]|1[0-2])').all():
        raise ValueError('year_month 格式錯誤')
    if frame.permno.isin(excluded).any(): raise ValueError('02_02 仍含已刪除 PERMNO')
    for month, permnos in valid.groupby('year_month')['permno']:
        monthly_nodes[month].update(permnos)
    linked = frame.loc[~missing_month & ~missing_thread & ~missing_permno]
    for (month, thread), permnos in linked.groupby(['year_month', 'thread_id'])['permno']:
        monthly_threads[month][thread].update(permnos)
if input_rows != parquet.metadata.num_rows or input_rows != upstream['after']['rows']:
    raise ValueError('來源筆數與 02_02 摘要不同')
if not monthly_nodes: raise ValueError('沒有有效月份及 PERMNO')
first_month, last_month = min(monthly_nodes), max(monthly_nodes)
targets = pd.period_range(pd.Period(first_month, freq='M') + WINDOW_MONTHS,
                          pd.Period(last_month, freq='M') + 1, freq='M')
if not len(targets): raise ValueError('資料不足六個月，無法建立完整視窗')
print(f'來源 {input_rows:,} 列；目標月份 {targets[0]} 至 {targets[-1]}，共 {len(targets)} 個網路')


來源 1,447,576 列；目標月份 2012-10 至 2020-09，共 96 個網路


In [3]:
def build_network(target, thread_months, node_months, window_months=6):
    target = pd.Period(target, freq='M')
    months = [str(target - offset) for offset in range(window_months, 0, -1)]
    threads = defaultdict(set)
    nodes = set()
    for month in months:
        nodes.update(node_months.get(month, set()))
        for thread, permnos in thread_months.get(month, {}).items():
            threads[thread].update(permnos)
    weights, thread_counts = Counter(), Counter()
    for permnos in threads.values():
        thread_counts.update(permnos)
        weights.update(combinations(sorted(permnos), 2))
    return months, nodes, threads, weights, thread_counts

# 驗證十月只使用 4–9 月、跨月 thread 合併、重複提及不加權、不同 thread 加權及孤立節點。
test_threads = {
    '2020-03': {'old': {99, 1}},
    '2020-04': {'t1': {1}, 't2': {1, 2}},
    '2020-09': {'t1': {1, 2}, 'solo': {3}},
    '2020-10': {'future': {88, 1}},
}
test_nodes = {m: set().union(*threads.values()) for m, threads in test_threads.items()}
test_nodes['2020-08'] = {5}
months, nodes, threads, weights, thread_counts = build_network('2020-10', test_threads, test_nodes)
assert months == ['2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09']
assert nodes == {1, 2, 3, 5}
assert weights == {(1, 2): 2} and thread_counts[1] == 2
assert build_network('2020-11', test_threads, test_nodes)[3][(1, 2)] == 1
print('Rolling window 與連結定義驗證通過')

# 別名已由 02 映射到同一 PERMNO，先合併再組合，不產生self-loop或重複共同thread。
alias_threads = {'2020-04': {'t1': {13159, 20000}}, '2020-09': {'t1': {13159, 20000}}}
alias_nodes = {month: set().union(*threads.values()) for month, threads in alias_threads.items()}
assert build_network('2020-10', alias_threads, alias_nodes)[3] == {(13159, 20000): 1}


Rolling window 與連結定義驗證通過


In [4]:
edge_schema = pa.schema([('permno_i', pa.int64()), ('permno_j', pa.int64()), ('n_common_threads', pa.int64()), ('weight', pa.float64())])
node_schema = pa.schema([('permno', pa.int64()), ('n_threads', pa.int64()), ('degree', pa.int64()), ('strength_raw', pa.int64()), ('strength', pa.float64())])
summaries = []
for index, target in enumerate(targets, 1):
    months, nodes, threads, weights, thread_counts = build_network(target, monthly_threads, monthly_nodes, WINDOW_MONTHS)
    degree, strength_raw, strength = Counter(), Counter(), Counter()
    edge_rows = []
    for (a, b), weight in sorted(weights.items()):
        if a >= b or weight < 1 or a not in nodes or b not in nodes:
            raise ValueError('無效的 edge')
        edge_rows.append({'permno_i': int(a), 'permno_j': int(b), 'n_common_threads': weight, 'weight': math.log1p(weight)})
        degree.update([a, b])
        strength_raw[a] += weight; strength_raw[b] += weight
        strength[a] += math.log1p(weight); strength[b] += math.log1p(weight)
    node_rows = [{'permno': int(permno), 'n_threads': thread_counts[permno],
                  'degree': degree[permno], 'strength_raw': strength_raw[permno], 'strength': strength[permno]} for permno in sorted(nodes)]
    expected_weight = sum(len(t) * (len(t) - 1) // 2 for t in threads.values())
    if sum(weights.values()) != expected_weight or sum(degree.values()) != 2 * len(weights):
        raise ValueError('Thread pair 與 network 統計不一致')
    if sum(strength_raw.values()) != 2 * sum(weights.values()): raise ValueError('Strength 統計不一致')
    if not math.isclose(sum(strength.values()), 2 * sum(math.log1p(w) for w in weights.values())):
        raise ValueError('Transformed strength reconciliation failed')
    pairs = [(row['permno_i'], row['permno_j']) for row in edge_rows]
    if len(set(pairs)) != len(pairs) or any(a >= b for a, b in pairs):
        raise ValueError('Self-loop, reversed or duplicate PERMNO pair')
    folder = OUT / str(target)
    folder.mkdir()
    for name, rows, schema in [('edges', edge_rows, edge_schema), ('nodes', node_rows, node_schema)]:
        temp = folder / (name + '.parquet.partial')
        table = pa.Table.from_pylist(rows, schema=schema)
        pq.write_table(table, temp, compression='zstd')
        if not pq.read_table(temp).equals(table): raise ValueError('輸出驗證失敗')
        temp.replace(folder / (name + '.parquet'))
    n = len(nodes)
    summaries.append({'network_month': str(target), 'window_start': months[0], 'window_end': months[-1],
                      'window_months': len(months), 'months_with_activity': sum(bool(monthly_nodes.get(m)) for m in months),
                      'n_nodes': n, 'n_edges': len(weights), 'n_threads': len(threads),
                      'n_linking_threads': sum(len(t) >= 2 for t in threads.values()),
                      'n_isolated_nodes': sum(degree[t] == 0 for t in nodes),
                      'total_edge_weight': sum(weights.values()),
                      'density': 2 * len(weights) / (n * (n - 1)) if n > 1 else 0.0})
    if index == 1 or index % 12 == 0 or index == len(targets):
        print(f'{index}/{len(targets)}：{target}，{n:,} nodes，{len(weights):,} edges', flush=True)
if source_stat != (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns): raise ValueError('來源執行中變動')
monthly_summary = pd.DataFrame(summaries)
monthly_summary.to_csv(OUT / 'monthly_network_summary.csv', index=False, encoding='utf-8-sig')
report = {'status': 'complete', 'source': str(SOURCE), 'source_sha256': source_hash,
          'node_unit':'permno', 'schema_version':2, 'upstream_manifest': str(MANIFEST), 'input_rows': input_rows,
          'source_month_min': first_month, 'source_month_max': last_month,
          'window_months': WINDOW_MONTHS, 'target_month_min': str(targets[0]), 'target_month_max': str(targets[-1]),
          'network_count': len(targets), 'missing_month_rows': missing_month_rows,
          'missing_thread_rows': missing_thread_rows, 'missing_permno_rows': missing_permno_rows,
          'edge_definition': 'Undirected distinct PERMNO pairs within each thread after filtering comments to [M-6, M-1]; no self loops.',
          'raw_weight_column': 'n_common_threads',
          'weight_definition': 'weight = log1p(n_common_threads); fixed within network month.',
          'raw_weight_definition': 'Number of distinct threads containing both PERMNOs within the window.',
          'node_definition': 'All PERMNOs mentioned in the window, including isolates and mentions without thread ID.',
          'output_directory': str(OUT), 'processing_seconds': time.perf_counter() - started}
(OUT / 'run_summary.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('完成：', OUT)
print(monthly_summary.head().to_string(index=False))


1/96：2012-10，259 nodes，1,898 edges
12/96：2013-09，199 nodes，822 edges
24/96：2014-09，383 nodes，2,817 edges
36/96：2015-09，910 nodes，14,492 edges
48/96：2016-09，1,428 nodes，47,249 edges
60/96：2017-09，1,329 nodes，53,480 edges
72/96：2018-09，1,399 nodes，60,905 edges
84/96：2019-09，1,681 nodes，267,878 edges
96/96：2020-09，2,352 nodes，914,402 edges
完成： c:\Users\eagle\RedditStock\Data\process\04_monthly_ticker_network\2026-10-13_05-59-46_UTC
network_month window_start window_end  window_months  months_with_activity  n_nodes  n_edges  n_threads  n_linking_threads  n_isolated_nodes  total_edge_weight  density
      2012-10      2012-04    2012-09              6                     6      259     1898        356                270                 7               3191 0.056808
      2012-11      2012-05    2012-10              6                     6      257     1806        356                265                 8               2982 0.054900
      2012-12      2012-06    2012-11              6        

## 描述統計：CRSP 股票母體、Reddit 覆蓋與 linkage 分布

- `N`：當月 CRSP 至少有一筆有效股票日期資料的不同 PERMNO 數。
- `stocks_with_redditlinkage`：上述 CRSP 母體中、當月 network degree > 0 的股票數。孤立節點不算有 linkage，非當月 CRSP 股票不進入覆蓋率分子。
- `coverage_pct`：有 linkage 的 CRSP 股票數／當月 CRSP 股票數 × 100。前面清理 ticker 的 CRSP 檔不作為母體，以免人為縮小分母。
- `redditlinkages_per_focal_firm`：所有 network 節點的 degree，包含孤立節點的 0；單位是 stock–network-month。同一股票可跨月重複出現。
- `common_threads_per_stockpair`：每個無方向 edge 的 raw n_common_threads；單位是 stock-pair–network-month，不使用 log 權重。

前面三項的整體分布以 network 月份為單位；後兩項另提供逐月分布，以及 pooled stock-month／pair-month 分布。每個分布皆報告 `N, mean, std, median, p90, min, max`；std 為樣本標準差，分位數採線性內插。來源 stock file 可透過 CRSP_STATS_INPUT 指定。

可單獨執行此段，不重建 network。結果存到該 network run 的新 statistics 時間目錄，舊資料保留。

Additional identical statistics restrict network_month to 2019 and 2020. Only existing network months are used; absent months are not filled with zeros.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import json, math, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

NETWORK_STATS_INPUT = None
CRSP_STATS_INPUT = None
ns_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Data/process').is_dir())
if NETWORK_STATS_INPUT is not None:
    ns_run = Path(NETWORK_STATS_INPUT)
    if not ns_run.is_absolute(): ns_run = ns_root/ns_run
elif 'OUT' in globals() and (Path(OUT)/'monthly_network_summary.csv').is_file():
    ns_run = Path(OUT)
else:
    ns_candidates = []
    for manifest in (ns_root/'Data/process/04_monthly_ticker_network').glob('*/run_summary.json'):
        info = json.loads(manifest.read_text(encoding='utf-8'))
        if info.get('status') == 'complete' and info.get('node_unit') == 'permno': ns_candidates.append(manifest.parent)
    if not ns_candidates: raise FileNotFoundError('請先成功執行新版 PERMNO network')
    ns_run = max(ns_candidates, key=lambda p:p.name)
ns_manifest = json.loads((ns_run/'run_summary.json').read_text(encoding='utf-8'))
assert ns_manifest['status']=='complete' and ns_manifest.get('node_unit')=='permno'
ns_crsp = Path(CRSP_STATS_INPUT) if CRSP_STATS_INPUT else ns_root/'Data/raw/stock/RAW_crsp.csv'
if not ns_crsp.is_absolute(): ns_crsp = ns_root/ns_crsp
ns_started = time.perf_counter()
ns_input_stats = {str(ns_crsp):(ns_crsp.stat().st_size,ns_crsp.stat().st_mtime_ns)}
ns_population = defaultdict(set)
ns_invalid_keys = 0
ns_header = pd.read_csv(ns_crsp,nrows=0).columns
ns_id = 'permno' if 'permno' in ns_header else 'PERMNO'
ns_date = 'DlyCalDt' if 'DlyCalDt' in ns_header else 'date'
for batch in pd.read_csv(ns_crsp,usecols=[ns_id,ns_date],dtype=str,keep_default_na=False,chunksize=250_000):
    ids = pd.to_numeric(batch[ns_id],errors='coerce')
    dates = pd.to_datetime(batch[ns_date],errors='coerce')
    valid = np.isfinite(ids) & ids.gt(0) & ids.mod(1).eq(0) & dates.notna()
    ns_invalid_keys += int((~valid).sum())
    pairs = pd.DataFrame({'month':dates[valid].dt.strftime('%Y-%m'),'permno':ids[valid].astype('int64')}).drop_duplicates()
    for month,values in pairs.groupby('month').permno: ns_population[month].update(values)

def ns_distribution(metric, frequency):
    # Integer histogram preserves exact pooled quantiles without storing millions of edges.
    items = sorted((value,count) for value,count in frequency.items() if count>0)
    count = sum(c for _,c in items)
    out = {'metric':metric,'N':count,**dict.fromkeys(['mean','std','median','p90','min','max'])}
    if not count: return out
    values = np.array([v for v,c in items],dtype=float)
    counts = np.array([c for v,c in items],dtype=np.int64)
    cumulative = counts.cumsum()
    mean = float(np.dot(values,counts)/count)
    def quantile(q):
        rank = q*(count-1)
        lower,upper = math.floor(rank),math.ceil(rank)
        a = values[np.searchsorted(cumulative,lower+1)]
        b = values[np.searchsorted(cumulative,upper+1)]
        return float(a+(b-a)*(rank-lower))
    out.update(mean=mean,std=float(np.sqrt(np.dot((values-mean)**2,counts)/(count-1))) if count>1 else None,
               median=quantile(.5),p90=quantile(.9),min=float(values[0]),max=float(values[-1]))
    return out

ns_monthly = []
ns_month_distributions = []
ns_degrees,ns_threads = Counter(),Counter()
ns_degrees_2019_2020,ns_threads_2019_2020 = Counter(),Counter()
ns_folders = sorted(p.parent for p in ns_run.glob('????-??/edges.parquet'))
assert len(ns_folders)==ns_manifest['network_count']
for index,folder in enumerate(ns_folders,1):
    month = folder.name
    for name in ['nodes.parquet','edges.parquet']:
        file = folder/name
        ns_input_stats[str(file)] = (file.stat().st_size,file.stat().st_mtime_ns)
    nodes = pd.read_parquet(folder/'nodes.parquet',columns=['permno','degree'])
    assert nodes.permno.is_unique and nodes.degree.ge(0).all()
    linked = set(nodes.loc[nodes.degree.gt(0),'permno'])
    population = ns_population.get(month,set())
    covered = len(linked & population)
    if not population: raise ValueError('當月 CRSP 母體為空，無法計算覆蓋率：'+month)
    degrees = Counter({int(v):int(c) for v,c in nodes.degree.value_counts().items()})
    threads = Counter()
    edge_count = 0
    for batch in pq.ParquetFile(folder/'edges.parquet').iter_batches(columns=['n_common_threads'],batch_size=200_000):
        values = batch.column(0).to_numpy()
        assert (values>0).all()
        unique,counts = np.unique(values,return_counts=True)
        threads.update({int(v):int(c) for v,c in zip(unique,counts)})
        edge_count += len(values)
    assert int(nodes.degree.sum())==2*edge_count
    ns_degrees.update(degrees);ns_threads.update(threads)
    if month[:4] in ['2019','2020']:
        ns_degrees_2019_2020.update(degrees);ns_threads_2019_2020.update(threads)
    ns_monthly.append({'network_month':month,'N':len(population),'stocks_with_redditlinkage':covered,
        'coverage_pct':100*covered/len(population),'network_nodes':len(nodes),
        'network_stocks_with_linkage':len(linked),'linked_nodes_outside_crsp_month':len(linked-population),
        'isolated_nodes':int(nodes.degree.eq(0).sum()),'n_edges':edge_count})
    for metric,freq in [('redditlinkages_per_focal_firm',degrees),('common_threads_per_stockpair',threads)]:
        ns_month_distributions.append({'network_month':month,**ns_distribution(metric,freq)})
    if index%24==0: print('network statistics:',index,'/',len(ns_folders),flush=True)
ns_monthly = pd.DataFrame(ns_monthly)
ns_overall = pd.DataFrame([
    ns_distribution('N_stocks_per_month',Counter(ns_monthly['N'])),
    ns_distribution('stocks_with_redditlinkage_per_month',Counter(ns_monthly.stocks_with_redditlinkage)),
    ns_distribution('coverage_pct_per_month',Counter(ns_monthly.coverage_pct)),
    ns_distribution('redditlinkages_per_focal_firm',ns_degrees),
    ns_distribution('common_threads_per_stockpair',ns_threads),
])
ns_selected = ns_monthly.loc[ns_monthly.network_month.str[:4].isin(['2019','2020'])].copy()
if ns_selected.empty: raise ValueError('No network months in 2019/2020')
ns_overall_2019_2020 = pd.DataFrame([
    ns_distribution('N_stocks_per_month',Counter(ns_selected['N'])),
    ns_distribution('stocks_with_redditlinkage_per_month',Counter(ns_selected.stocks_with_redditlinkage)),
    ns_distribution('coverage_pct_per_month',Counter(ns_selected.coverage_pct)),
    ns_distribution('redditlinkages_per_focal_firm',ns_degrees_2019_2020),
    ns_distribution('common_threads_per_stockpair',ns_threads_2019_2020),
])
assert sum(ns_degrees_2019_2020.values()) == int(ns_selected.network_nodes.sum())
assert sum(ns_threads_2019_2020.values()) == int(ns_selected.n_edges.sum())
for file,stat in ns_input_stats.items():
    assert stat==(Path(file).stat().st_size,Path(file).stat().st_mtime_ns)
ns_out = ns_run/'statistics'/datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_UTC')
ns_out.mkdir(parents=True,exist_ok=False)
ns_monthly.to_csv(ns_out/'monthly_coverage.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(ns_month_distributions).to_csv(ns_out/'monthly_linkage_distributions.csv',index=False,encoding='utf-8-sig')
ns_overall.to_csv(ns_out/'network_descriptive_statistics.csv',index=False,encoding='utf-8-sig')
ns_selected.to_csv(ns_out/'monthly_coverage_2019_2020.csv',index=False,encoding='utf-8-sig')
ns_selected_distributions = pd.DataFrame(ns_month_distributions)
ns_selected_distributions = ns_selected_distributions.loc[ns_selected_distributions.network_month.str[:4].isin(['2019','2020'])]
ns_selected_distributions.to_csv(ns_out/'monthly_linkage_distributions_2019_2020.csv',index=False,encoding='utf-8-sig')
ns_overall_2019_2020.to_csv(ns_out/'network_descriptive_statistics_2019_2020.csv',index=False,encoding='utf-8-sig')
ns_report = {'status':'complete','network_run':str(ns_run),'crsp_source':str(ns_crsp),
    'restricted_years':[2019,2020], 'restricted_months':len(ns_selected),
    'restricted_month_min':ns_selected.network_month.min(), 'restricted_month_max':ns_selected.network_month.max(),
    'distribution_2019_2020':ns_overall_2019_2020.to_dict(orient='records'),
    'months':len(ns_monthly),'invalid_crsp_key_rows':ns_invalid_keys,
    'stock_population':'distinct PERMNO with at least one valid CRSP observation that calendar month',
    'covered_stocks':'CRSP population intersect network nodes with degree > 0; excludes isolates',
    'focal_firm_population':'all network node-months, including isolates; not restricted to CRSP intersection',
    'pair_population':'all undirected edge-months; raw n_common_threads',
    'std_ddof':1,'quantile_method':'linear; exact frequency histogram',
    'distribution':ns_overall.to_dict(orient='records'),'runtime_seconds':time.perf_counter()-ns_started}
(ns_out/'run_summary.json').write_text(json.dumps(ns_report,ensure_ascii=False,indent=2),encoding='utf-8')
display(ns_overall)
display(ns_overall_2019_2020)
print('04 統計輸出：',ns_out)
